# Session 2 : les exercices expliqués pas à pas

**Le neurone sigmoïde : la régression logistique** (MSA-DATB09-03, Neural Networks & Deep Learning Foundations)

Ce notebook reprend les **3 exercices** de la session 2 (slides 6, 10 et 16) :

| Exercice | Sujet | Ce qu'on apprend |
|---|---|---|
| 1 | La sigmoïde et un forward pass | transformer un score en probabilité |
| 2 | Trois log loss à la main | mesurer à quel point une prédiction est mauvaise |
| 3 | Un pas de descente de gradient | corriger les poids pour faire baisser l'erreur |

**Comment le lire.** Pour chaque question, tu trouveras :

1. 📖 **La formule** : ce qu'elle veut dire, symbole par symbole, et pourquoi on l'utilise.
2. ✍️ **Le calcul à la main** : exactement ce que tu écrirais sur ta copie, avec les valeurs approchées de l'énoncé.
3. 💻 **La vérification en Python** : on refait le calcul avec NumPy pour être sûr du résultat.

Exécute les cellules **dans l'ordre**, de haut en bas (Shift + Entrée).

## 0. Le vocabulaire (à garder sous les yeux)

| Symbole | Nom | Ce que c'est, en clair |
|---|---|---|
| $x$ | l'exemple (les entrées) | les mesures d'un objet. Ici 2 nombres, par exemple la largeur et la longueur des feuilles d'une plante : $x = (x_1, x_2)$ |
| $y$ | l'étiquette (le label) | la vraie réponse : 1 (plante toxique) ou 0 (plante inoffensive) |
| $w$ | les poids | un nombre par mesure. Il dit **combien** la mesure compte et **dans quel sens** (positif : pousse vers la classe 1, négatif : pousse vers la classe 0) |
| $b$ | le biais | un décalage constant, ajouté à la fin |
| $z$ | le score (ou « logit ») | $z = w^\top x + b$ : la somme pondérée des mesures. N'importe quel nombre, négatif ou positif |
| $\sigma$ | la sigmoïde | la fonction qui transforme $z$ en un nombre entre 0 et 1 |
| $a$ | l'activation (la sortie) | $a = \sigma(z)$ : la **probabilité** que l'exemple soit de classe 1 |
| $\mathcal{L}$ | la loss (perte) | l'erreur sur **un** exemple |
| $J$ | le coût | la **moyenne** des loss sur tous les exemples |
| $m$ | | le nombre d'exemples |
| $\eta$ | le learning rate (« éta ») | la taille du pas quand on corrige les poids |

Le trajet d'un exemple dans le neurone :

```
 x  --->  z = w.x + b  --->  a = sigmoid(z)  --->  L(a, y)
entrées      score            probabilité          erreur
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Afficher les tableaux NumPy avec 4 décimales, c'est plus lisible
np.set_printoptions(precision=4, suppress=True)

---
# Exercice 1 : la sigmoïde et un forward pass

> **Énoncé (slide 6)**
>
> 1. Calculer $\sigma(0)$, $\sigma(2)$, $\sigma(-2)$ et $\sigma(4)$. Utiliser $e^{-2} \approx 0.135$ et $e^{-4} \approx 0.018$. Vérifier la symétrie $\sigma(-2) = 1 - \sigma(2)$.
> 2. Un neurone a $w = (2, -1)$ et $b = -1$. Pour l'exemple $x = (1, 3)$, calculer $z$, puis $a = \sigma(z)$.
> 3. Que vaut $P(y = 1 \mid x)$ ? Quelle classe est prédite ?

## 📖 La formule : la sigmoïde

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Pourquoi on en a besoin ?** Le perceptron de la session 1 utilisait une « marche » : il répondait 0 ou 1, point. Deux problèmes :

- **pas de confiance** : un score $z = 0.01$ (à peine positif) et un score $z = 100$ (énorme) donnaient tous les deux 1 ;
- **pas de pente** : la marche est plate partout, donc impossible de savoir dans quel sens corriger les poids (pas de descente de gradient possible).

La sigmoïde est une **marche adoucie** : elle passe en douceur de 0 à 1. Elle prend n'importe quel nombre et le « presse » entre 0 et 1, ce qui permet de lire le résultat comme une **probabilité**.

**Les symboles :**

- $e \approx 2.718$ est une constante mathématique (comme $\pi \approx 3.14$).
- $e^{-z}$ se lit « e puissance moins z ». Une puissance négative, c'est l'inverse : $e^{-2} = \dfrac{1}{e^2}$.
- Tout nombre puissance 0 vaut 1 : $e^0 = 1$.

**La recette, toujours la même, en 3 étapes :**

1. calculer $e^{-z}$ ;
2. ajouter 1 ;
3. faire 1 divisé par le résultat.

**Pourquoi le résultat est toujours entre 0 et 1 ?** $e^{-z}$ est toujours strictement positif, donc $1 + e^{-z} > 1$, donc $\dfrac{1}{1 + e^{-z}}$ est entre 0 et 1.

- si $z$ est très grand, $e^{-z}$ est minuscule, le dénominateur vaut presque 1, donc $\sigma(z) \approx 1$ ;
- si $z$ est très négatif, $e^{-z}$ est énorme, donc $\sigma(z) \approx 0$.

## ✍️ Question 1 : le calcul à la main

**$\sigma(0)$**

1. $e^{-0} = e^0 = 1$
2. $1 + 1 = 2$
3. $\sigma(0) = \dfrac{1}{2} = \mathbf{0.5}$

➡️ Un score nul veut dire que le modèle hésite complètement : 50 / 50.

**$\sigma(2)$**

1. $e^{-2} \approx 0.135$ (donné par l'énoncé)
2. $1 + 0.135 = 1.135$
3. $\sigma(2) = \dfrac{1}{1.135} \approx \mathbf{0.881}$

**$\sigma(4)$**

1. $e^{-4} \approx 0.018$ (donné par l'énoncé)
2. $1 + 0.018 = 1.018$
3. $\sigma(4) = \dfrac{1}{1.018} \approx \mathbf{0.982}$

**$\sigma(-2)$** ⚠️ *attention au signe, c'est le piège classique*

1. Ici $-z = -(-2) = +2$, il faut donc $e^{+2}$, qui n'est **pas** donné. Astuce : $e^{2} = \dfrac{1}{e^{-2}} \approx \dfrac{1}{0.135} \approx 7.41$
2. $1 + 7.41 = 8.41$
3. $\sigma(-2) = \dfrac{1}{8.41} \approx \mathbf{0.119}$

*Méthode plus rapide* : multiplier en haut et en bas par $e^{-2}$ pour n'utiliser que la valeur donnée :

$$\sigma(-2) = \frac{1}{1 + e^{2}} = \frac{e^{-2}}{e^{-2} + 1} \approx \frac{0.135}{1.135} \approx 0.119$$

**Vérification de la symétrie**

$$1 - \sigma(2) \approx 1 - 0.881 = 0.119 = \sigma(-2) \quad ✓$$

**Pourquoi c'est vrai pour n'importe quel $z$ ?** La preuve tient en deux lignes :

$$1 - \sigma(z) = 1 - \frac{1}{1 + e^{-z}} = \frac{1 + e^{-z} - 1}{1 + e^{-z}} = \frac{e^{-z}}{1 + e^{-z}}$$

On multiplie en haut et en bas par $e^{z}$ (et $e^{-z} \times e^{z} = e^0 = 1$) :

$$\frac{e^{-z}}{1 + e^{-z}} = \frac{1}{e^{z} + 1} = \sigma(-z)$$

En image : la courbe est symétrique autour du point $(0\,;\,0.5)$. Ce que $z = +2$ ajoute au-dessus de 0.5 (+0.381), $z = -2$ l'enlève en dessous (-0.381).

## 💻 Vérification en Python

On code la sigmoïde exactement comme la formule. On utilise `np.exp` (et pas `math.exp`) parce que `np.exp` fonctionne aussi sur un **tableau entier**, valeur par valeur : on pourra calculer les probabilités de 1000 exemples d'un coup.

In [ ]:
def sigmoid(z):
    # sigma(z) = 1 / (1 + e^(-z)), marche sur un nombre ou sur un tableau
    return 1 / (1 + np.exp(-z))


# Les valeurs "à la main", avec les approximations de l'énoncé
a_la_main = {0: 1 / (1 + 1),
             2: 1 / (1 + 0.135),
             -2: 0.135 / (1 + 0.135),
             4: 1 / (1 + 0.018)}

print(f"{'z':>4} | {'à la main':>10} | {'exact (NumPy)':>13}")
print("-" * 35)
for z in [0, 2, -2, 4]:
    print(f"{z:>4} | {a_la_main[z]:>10.4f} | {sigmoid(z):>13.4f}")

print()
print("Symétrie : 1 - sigma(2) =", round(1 - sigmoid(2), 4), "  et  sigma(-2) =", round(sigmoid(-2), 4))

Les petites différences à la 4e décimale viennent des arrondis de l'énoncé ($e^{-2} \approx 0.135$ au lieu de $0.1353...$). Arrondis à 3 décimales, les résultats sont identiques.

Voyons la courbe, avec nos 4 points en rouge et la « marche » du perceptron en pointillés pour comparer :

In [ ]:
z_grille = np.linspace(-6, 6, 300)          # 300 valeurs de z entre -6 et 6

plt.figure(figsize=(7, 4))
plt.plot(z_grille, sigmoid(z_grille), label="sigmoïde σ(z)")
plt.plot(z_grille, (z_grille >= 0).astype(float), "--", color="gray", label="marche (perceptron)")
for z0 in [-2, 0, 2, 4]:
    plt.scatter(z0, sigmoid(z0), color="red", zorder=3)
    plt.annotate(f"σ({z0}) = {sigmoid(z0):.3f}", (z0, sigmoid(z0)),
                 textcoords="offset points", xytext=(8, -14))
plt.axhline(0.5, color="lightgray", lw=0.8)
plt.axvline(0, color="lightgray", lw=0.8)
plt.xlabel("z (le score)")
plt.ylabel("a = σ(z) (la probabilité)")
plt.title("La sigmoïde : une marche adoucie")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 📖 Question 2 : la formule du forward pass

$$z = w^\top x + b = w_1 x_1 + w_2 x_2 + b \qquad \text{puis} \qquad a = \sigma(z)$$

$w^\top x$ se lit « w transposé x » (on dit aussi « produit scalaire de $w$ et $x$ »). Ça veut juste dire : **on multiplie chaque poids par la mesure qui lui correspond, puis on additionne tout**. Ensuite on ajoute le biais $b$.

Enchaîner ces deux étapes s'appelle le **forward pass** (« passe avant ») : on part de l'entrée $x$ et on avance jusqu'à la sortie $a$.

**Lire les données de l'exercice :**

- $x = (1, 3)$ : $x_1 = 1$ et $x_2 = 3$ ;
- $w = (2, -1)$ : $w_1 = 2$ est positif, donc $x_1$ pousse vers la classe 1 ; $w_2 = -1$ est négatif, donc $x_2$ pousse vers la classe 0 ;
- $b = -1$.

## ✍️ Le calcul à la main

$$z = w_1 x_1 + w_2 x_2 + b = 2 \times 1 + (-1) \times 3 + (-1) = 2 - 3 - 1 = \mathbf{-2}$$

$$a = \sigma(-2) \approx \mathbf{0.119}$$

On l'a déjà calculé à la question 1 : l'exercice est construit exprès pour réutiliser ce résultat.

In [ ]:
w = np.array([2.0, -1.0])    # les poids
b = -1.0                     # le biais
x = np.array([1.0, 3.0])     # l'exemple

# Version "à la main", terme par terme
z_detail = w[0] * x[0] + w[1] * x[1] + b
print("z (terme par terme) =", z_detail)

# Version NumPy : @ fait le produit scalaire (multiplier terme à terme puis additionner)
z = w @ x + b
print("z (avec @)          =", z)

a = sigmoid(z)
print("a = sigma(z)        =", round(a, 4))

## 📖 Question 3 : probabilité et classe prédite

**La formule.** Par construction du modèle (c'est exactement ce qu'on appelle la **régression logistique** : un neurone avec une sigmoïde) :

$$a = \sigma(w^\top x + b) = P(y = 1 \mid x)$$

On lit $P(y = 1 \mid x)$ : « la probabilité que la classe soit 1, **sachant** l'exemple $x$ ». La barre verticale veut dire « sachant ».

**La règle de décision** : on prédit la classe 1 si $a \ge 0.5$, sinon la classe 0.

**Le raccourci** : la sigmoïde est croissante et $\sigma(0) = 0.5$, donc

$$a \ge 0.5 \iff \sigma(z) \ge \sigma(0) \iff z \ge 0$$

Il suffit de regarder **le signe de $z$** ! La sigmoïde ne change pas la décision, elle ajoute le **niveau de confiance**. C'est aussi pour ça que la frontière de décision reste la droite $w^\top x + b = 0$, comme pour le perceptron.

## ✍️ La réponse

- $P(y = 1 \mid x) = a \approx 0.119$, soit environ **12 %**.
- $0.119 < 0.5$ (ou plus simplement $z = -2 < 0$), donc **classe 0 prédite**.
- Le modèle est plutôt sûr de lui : $P(y = 0 \mid x) = 1 - a \approx 0.881$, soit **88 %** pour la classe 0.

In [ ]:
p_classe_1 = a
p_classe_0 = 1 - a
classe_predite = int(a >= 0.5)      # True devient 1, False devient 0

print(f"P(y = 1 | x) = {p_classe_1:.3f}  (soit {p_classe_1:.0%})")
print(f"P(y = 0 | x) = {p_classe_0:.3f}  (soit {p_classe_0:.0%})")
print("Classe prédite :", classe_predite)
print("Raccourci : z =", z, "est négatif, donc classe", int(z >= 0))

> ✅ **À retenir (exercice 1)**
>
> - $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ est toujours entre 0 et 1 ; $\sigma(0) = 0.5$ ; $\sigma(-z) = 1 - \sigma(z)$.
> - Forward pass : $z = w^\top x + b$, puis $a = \sigma(z)$.
> - $a$ est la probabilité de la classe 1. On prédit 1 si $a \ge 0.5$, c'est-à-dire si $z \ge 0$.
> - Plus $z$ est loin de 0, plus le modèle est confiant ($z = 2$ donne 88 %, $z = 4$ donne 98 %).

---
# Exercice 2 : trois log loss à la main

> **Énoncé (slide 10)**
>
> Trois exemples, tous avec la vraie étiquette $y = 1$. Le modèle prédit $a = 0.9$, $a = 0.5$ et $a = 0.1$.
>
> 1. Calculer la loss $\mathcal{L}$ de chaque exemple (logarithme naturel).
> 2. Calculer le coût $J$ de ces 3 exemples.
> 3. Quel exemple contribue le plus à $J$ ? Que devient sa loss si $a = 0.01$ ?

## 📖 La formule : la log loss (ou « binary cross-entropy »)

$$\mathcal{L}(a, y) = -\big[\, y \log a + (1 - y)\log(1 - a) \,\big] \qquad\qquad J = \frac{1}{m}\sum_{i=1}^{m} \mathcal{L}^{(i)}$$

$\mathcal{L}$ est l'erreur sur **un** exemple, $J$ est la **moyenne** sur les $m$ exemples. Le petit $(i)$ en exposant veut dire « l'exemple numéro $i$ » (ce n'est pas une puissance).

**D'où vient cette formule (version courte) :**

1. Le modèle dit : « probabilité de la classe 1 = $a$ », donc « probabilité de la classe 0 = $1 - a$ ».
2. Un bon modèle donne une **grande probabilité à la vraie réponse**. Les deux cas s'écrivent en une seule formule : $P(y \mid x) = a^{y}(1 - a)^{1 - y}$. Essaie : avec $y = 1$ il reste $a^1 (1-a)^0 = a$ ; avec $y = 0$ il reste $a^0 (1-a)^1 = 1 - a$.
3. Sur tout le dataset, on multiplie ces probabilités : c'est la **vraisemblance** (likelihood). On veut la **maximiser**.
4. On prend le **log** : il transforme le produit en somme (plus simple), et évite qu'un produit de 1000 nombres plus petits que 1 devienne 0 dans l'ordinateur. Le log est croissant, donc le maximum reste au même endroit.
5. On met un **signe moins** pour obtenir une erreur à **minimiser**, et on divise par $m$ pour avoir une moyenne.

**Le truc qui simplifie tout** : $y$ vaut 0 ou 1, donc **un seul des deux termes est actif** :

- si $y = 1$ : $(1 - y) = 0$ efface le 2e terme, il reste $\mathcal{L} = -\log a$ ;
- si $y = 0$ : $y = 0$ efface le 1er terme, il reste $\mathcal{L} = -\log(1 - a)$.

**Rappels sur le logarithme naturel** (noté $\log$ ou $\ln$, c'est la touche « ln » de la calculatrice) :

- $\log 1 = 0$ : une prédiction parfaite ($a = 1$ quand $y = 1$) ne coûte rien ;
- le log d'un nombre entre 0 et 1 est **négatif**, d'où le signe moins devant : la loss est toujours **positive** ;
- quand $a$ se rapproche de 0, $\log a$ plonge vers $-\infty$ : la loss explose.

## ✍️ Question 1 : les 3 loss à la main

Les trois exemples ont $y = 1$, donc $\mathcal{L} = -\log a$ :

| Exemple | $a$ prédit | Calcul | $\mathcal{L}$ | Commentaire |
|---|---|---|---|---|
| 1 | 0.9 | $-\log(0.9)$ | **0.105** | bonne réponse, confiante : presque gratuit |
| 2 | 0.5 | $-\log(0.5)$ | **0.693** | le modèle hésite : coût moyen |
| 3 | 0.1 | $-\log(0.1)$ | **2.303** | mauvaise réponse, confiante : très cher |

À la calculatrice : $\ln(0.9) \approx -0.105$, $\ln(0.5) \approx -0.693$, $\ln(0.1) \approx -2.303$. On enlève le signe moins.

## ✍️ Question 2 : le coût $J$

Le coût est la **moyenne** des loss :

$$J = \frac{0.105 + 0.693 + 2.303}{3} = \frac{3.101}{3} \approx \mathbf{1.034}$$

## ✍️ Question 3 : quel exemple pèse le plus ?

- C'est l'**exemple 3** ($a = 0.1$) : sa loss de 2.303 représente à elle seule $\dfrac{2.303}{3.101} \approx 74\ \%$ du total.
- Si $a = 0.01$ : $\mathcal{L} = -\log(0.01) \approx \mathbf{4.605}$, soit **2 fois plus**.

**Pourquoi exactement 2 fois ?** $\log(0.01) = \log(0.1^2) = 2\log(0.1)$. Chaque fois qu'on divise $a$ par 10, la loss augmente de $\log 10 \approx 2.303$. Et si $a$ tend vers 0, la loss tend vers l'infini.

➡️ **L'idée clé** : la log loss punit très fort une erreur **commise avec assurance**. Un modèle qui dit « 1 % de chances » alors que c'est vrai se fait lourdement sanctionner. Pendant l'entraînement, ce sont donc ces exemples-là qui seront corrigés en priorité.

In [ ]:
def loss_un_exemple(a, y):
    # Log loss : -[y log(a) + (1 - y) log(1 - a)]. np.log est le logarithme naturel (ln)
    return -(y * np.log(a) + (1 - y) * np.log(1 - a))


a_pred = np.array([0.9, 0.5, 0.1])   # les 3 prédictions du modèle
y_vrai = np.array([1, 1, 1])         # les 3 vraies étiquettes

L = loss_un_exemple(a_pred, y_vrai)  # NumPy calcule les 3 loss d'un coup
for i in range(3):
    print(f"exemple {i + 1} : a = {a_pred[i]}  ->  L = {L[i]:.3f}  ({L[i] / L.sum():.0%} du total)")

J = L.mean()                         # le coût = la moyenne
print(f"\nCoût J = ({L[0]:.3f} + {L[1]:.3f} + {L[2]:.3f}) / 3 = {J:.4f}")

print(f"\nSi a = 0.01 : L = {loss_un_exemple(0.01, 1):.3f}  (contre {loss_un_exemple(0.1, 1):.3f} pour a = 0.1)")

Sur la courbe, on voit bien que la loss reste faible tant que la prédiction est bonne, puis grimpe en flèche quand le modèle se trompe avec assurance :

In [ ]:
a_grille = np.linspace(0.001, 0.999, 500)

plt.figure(figsize=(7, 4))
plt.plot(a_grille, -np.log(a_grille), label="si y = 1 : L = -log(a)")
plt.plot(a_grille, -np.log(1 - a_grille), label="si y = 0 : L = -log(1 - a)")
for a0 in [0.9, 0.5, 0.1, 0.01]:
    plt.scatter(a0, -np.log(a0), color="red", zorder=3)
    plt.annotate(f"a = {a0} : L = {-np.log(a0):.2f}", (a0, -np.log(a0)),
                 textcoords="offset points", xytext=(8, 4))
plt.ylim(0, 5)
plt.xlabel("probabilité prédite a")
plt.ylabel("loss L")
plt.title("La log loss punit les erreurs confiantes")
plt.legend(loc="center", bbox_to_anchor=(0.62, 0.62))
plt.grid(alpha=0.3)
plt.show()

> ✅ **À retenir (exercice 2)**
>
> - $\mathcal{L} = -[y \log a + (1 - y)\log(1 - a)]$ : avec $y = 1$ il reste $-\log a$, avec $y = 0$ il reste $-\log(1 - a)$.
> - $J$ est la moyenne des $\mathcal{L}$.
> - Bonne prédiction confiante : loss proche de 0. Mauvaise prédiction confiante : loss énorme (infinie à la limite).

---
# Exercice 3 : un pas de descente de gradient à la main

**Énoncé (slide 16)**

| | $x_1$ | $x_2$ | $y$ |
|---|---|---|---|
| exemple 1 | 2 | 1 | 1 |
| exemple 2 | 0 | 2 | 0 |

Départ : $w = (0, 0)$, $b = 0$. Learning rate $\eta = 2$. Utiliser $e^{-1.5} \approx 0.223$ et $e^{1} \approx 2.718$.

1. Forward : $a^{(1)}$, $a^{(2)}$ et le coût $J$.
2. Gradients : $dw = \frac{1}{2}X^\top(A - y)$ et $db$.
3. Mettre à jour $w$ et $b$.
4. Nouveau forward pass et nouveau coût. Est-ce que $J$ a diminué ?

## 📖 L'idée : la descente de gradient

Imagine que tu es sur une colline dans le brouillard et que tu veux descendre au point le plus bas. Tu ne vois rien au loin, mais tu sens **la pente sous tes pieds**. Stratégie : faire un petit pas **dans le sens où ça descend**, puis recommencer.

- La « colline », c'est le coût $J(w, b)$ : on veut le rendre le plus petit possible.
- La « pente », c'est le **gradient** : $\dfrac{\partial J}{\partial w}$ (noté $dw$) et $\dfrac{\partial J}{\partial b}$ (noté $db$). Il indique dans quel sens $J$ **monte** quand on augmente un peu $w$ ou $b$.
- On va donc dans le sens **opposé**, d'où le signe moins :

$$w \leftarrow w - \eta\, dw \qquad\qquad b \leftarrow b - \eta\, db$$

- La flèche $\leftarrow$ veut dire « remplacer par » : le nouveau $w$ est l'ancien $w$ moins $\eta\,dw$.
- $\eta$ est la **taille du pas** : trop petit, c'est lent ; trop grand, on saute par-dessus le minimum.
- Le symbole $\partial$ (« d rond ») veut dire « dérivée partielle » : la pente par rapport à une seule variable, en gardant toutes les autres fixes.

## 📖 Les formules du gradient, et d'où elles viennent

Pour un exemple, le calcul est une chaîne : $w_j \to z \to a \to \mathcal{L}$. La **règle de la chaîne** dit que pour connaître l'effet de $w_j$ sur $\mathcal{L}$, on **multiplie les pentes de chaque maillon** :

$$\frac{\partial \mathcal{L}}{\partial w_j} = \frac{\partial \mathcal{L}}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_j}$$

| Maillon | Pente | Pourquoi |
|---|---|---|
| $\frac{\partial z}{\partial w_j}$ | $x_j$ | $z = w_1 x_1 + w_2 x_2 + b$ : si $w_1$ augmente de 1, $z$ augmente de $x_1$ (comme la dérivée de $3w$ est $3$) |
| $\frac{\partial a}{\partial z}$ | $a(1 - a)$ | la dérivée de la sigmoïde : $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$ (slide 13) |
| $\frac{\partial \mathcal{L}}{\partial a}$ | $\frac{a - y}{a(1 - a)}$ | dérivée de $-[y\log a + (1-y)\log(1-a)]$, avec la règle $(\log u)' = \frac{1}{u}$ : on obtient $-\frac{y}{a} + \frac{1 - y}{1 - a}$, puis on met au même dénominateur |

On multiplie les trois : les $a(1 - a)$ en haut et en bas **se simplifient**, et il reste une formule très simple :

$$\frac{\partial \mathcal{L}}{\partial w_j} = (a - y)\, x_j \qquad\qquad \frac{\partial \mathcal{L}}{\partial b} = a - y \quad (\text{car } \tfrac{\partial z}{\partial b} = 1)$$

En mots : **le gradient, c'est l'erreur multipliée par l'entrée**. L'erreur $a - y$ est l'écart entre ce que le modèle a dit et la vérité.

**Sur tout le dataset** ($m$ exemples), $J$ est la moyenne des loss, donc son gradient est la moyenne des gradients :

$$dw = \frac{1}{m} X^\top (A - y) \qquad\qquad db = \frac{1}{m}\sum_{i=1}^{m}\big(a^{(i)} - y^{(i)}\big)$$

- $X$ est le tableau des entrées, **une ligne par exemple** (forme $(m, n)$ : $m$ exemples, $n$ mesures) ;
- $A$ est le vecteur des $m$ prédictions, $y$ celui des $m$ vraies étiquettes, donc $A - y$ est le vecteur des **erreurs** ;
- $X^\top (A - y)$ est juste une écriture compacte de $\sum_i (a^{(i)} - y^{(i)})\, x^{(i)}$ : la somme des « erreur × entrée » de chaque exemple.

## ✍️ Question 1 : forward pass et coût de départ

Avec $w = (0, 0)$ et $b = 0$, le score est nul pour tout le monde :

- exemple 1 : $z^{(1)} = 0 \times 2 + 0 \times 1 + 0 = 0$, donc $a^{(1)} = \sigma(0) = \mathbf{0.5}$
- exemple 2 : $z^{(2)} = 0 \times 0 + 0 \times 2 + 0 = 0$, donc $a^{(2)} = \sigma(0) = \mathbf{0.5}$

Le modèle de départ ne sait rien : il répond 50 / 50 partout.

Les loss (on applique le « truc » de l'exercice 2) :

- exemple 1, $y = 1$ : $\mathcal{L}^{(1)} = -\log a^{(1)} = -\log 0.5 \approx 0.693$
- exemple 2, $y = 0$ : $\mathcal{L}^{(2)} = -\log(1 - a^{(2)}) = -\log 0.5 \approx 0.693$

$$J = \frac{0.693 + 0.693}{2} = \log 2 \approx \mathbf{0.693}$$

## ✍️ Question 2 : les gradients

**Étape 1, les erreurs** $A - y$ :

$$A - y = \begin{pmatrix} 0.5 - 1 \\ 0.5 - 0 \end{pmatrix} = \begin{pmatrix} -0.5 \\ +0.5 \end{pmatrix}$$

Lecture : l'exemple 1 est **sous-estimé** (on a dit 0.5, il fallait 1), l'exemple 2 est **surestimé** (on a dit 0.5, il fallait 0).

**Étape 2, erreur × entrée pour chaque exemple** :

- exemple 1 : $(-0.5) \times (2, 1) = (-1,\ -0.5)$
- exemple 2 : $(+0.5) \times (0, 2) = (0,\ 1)$
- somme : $(-1 + 0,\ -0.5 + 1) = (-1,\ 0.5)$

**Étape 3, diviser par $m = 2$** :

$$dw = \frac{1}{2}\,(-1,\ 0.5) = \mathbf{(-0.5,\ 0.25)}$$

*La même chose avec les matrices.* $X = \begin{pmatrix} 2 & 1 \\ 0 & 2 \end{pmatrix}$ (une ligne par exemple). Transposer, c'est échanger lignes et colonnes : $X^\top = \begin{pmatrix} 2 & 0 \\ 1 & 2 \end{pmatrix}$. Puis on multiplie chaque ligne de $X^\top$ par le vecteur des erreurs :

$$X^\top (A - y) = \begin{pmatrix} 2 \times (-0.5) + 0 \times 0.5 \\ 1 \times (-0.5) + 2 \times 0.5 \end{pmatrix} = \begin{pmatrix} -1 \\ 0.5 \end{pmatrix}$$

C'est bien le même résultat qu'à l'étape 2.

**Le biais** : la moyenne des erreurs

$$db = \frac{1}{2}\big((-0.5) + 0.5\big) = \mathbf{0}$$

Les deux erreurs se compensent, donc le biais ne bougera pas.

## ✍️ Question 3 : la mise à jour

$$w \leftarrow w - \eta\, dw = (0, 0) - 2 \times (-0.5,\ 0.25) = (0 + 1,\ 0 - 0.5) = \mathbf{(1,\ -0.5)}$$

$$b \leftarrow b - \eta\, db = 0 - 2 \times 0 = \mathbf{0}$$

**Est-ce logique ?**

- $w_1$ monte (de 0 à 1) : $x_1$ vaut 2 pour l'exemple de classe 1 et 0 pour celui de classe 0. Donner un poids positif à $x_1$ pousse l'exemple 1 vers la classe 1. ✓
- $w_2$ descend (de 0 à -0.5) : $x_2$ est plus grand pour l'exemple de classe 0 (2 contre 1). Un poids négatif pousse l'exemple 2 vers la classe 0. ✓

## ✍️ Question 4 : nouveau forward pass et nouveau coût

Avec $w = (1, -0.5)$ et $b = 0$ :

**Exemple 1** : $z^{(1)} = 1 \times 2 + (-0.5) \times 1 + 0 = 1.5$

$$a^{(1)} = \sigma(1.5) = \frac{1}{1 + e^{-1.5}} \approx \frac{1}{1 + 0.223} = \frac{1}{1.223} \approx \mathbf{0.818}$$

**Exemple 2** : $z^{(2)} = 1 \times 0 + (-0.5) \times 2 + 0 = -1$

$$a^{(2)} = \sigma(-1) = \frac{1}{1 + e^{1}} \approx \frac{1}{1 + 2.718} = \frac{1}{3.718} \approx \mathbf{0.269}$$

(Attention au signe : $\sigma(-1)$ fait apparaître $e^{-(-1)} = e^{+1}$, c'est pour ça que l'énoncé donne $e^1$.)

**Nouveau coût** :

- exemple 1 ($y = 1$) : $\mathcal{L}^{(1)} = -\log(0.818) \approx 0.201$
- exemple 2 ($y = 0$) : $\mathcal{L}^{(2)} = -\log(1 - 0.269) = -\log(0.731) \approx 0.313$

$$J = \frac{0.201 + 0.313}{2} \approx \mathbf{0.257}$$

**Oui, $J$ a diminué : de 0.693 à 0.257**, presque divisé par 3 en un seul pas. Les deux prédictions se sont rapprochées de la vérité :

- exemple 1 : $a$ passe de 0.5 à 0.818 (la vraie réponse est 1) ;
- exemple 2 : $a$ passe de 0.5 à 0.269 (la vraie réponse est 0).

## 💻 Vérification en Python

On refait les 4 questions avec NumPy. Chaque ligne de code correspond à une formule du cours.

In [ ]:
# Les données : une ligne par exemple
X = np.array([[2.0, 1.0],     # exemple 1 : x1 = 2, x2 = 1
              [0.0, 2.0]])    # exemple 2 : x1 = 0, x2 = 2
y = np.array([1.0, 0.0])      # les vraies étiquettes
m = len(y)                    # nombre d'exemples (2)

w = np.array([0.0, 0.0])      # poids de départ
b = 0.0                       # biais de départ
eta = 2                       # learning rate


def cout(A, y):
    # J = moyenne des log loss (fonction de l'exercice 2)
    return loss_un_exemple(A, y).mean()


# 1. Forward pass
Z = X @ w + b                 # les 2 scores d'un coup, forme (m,)
A = sigmoid(Z)                # les 2 probabilités
print("Question 1 : forward")
print("  z =", Z, "  A =", A, "  J =", round(cout(A, y), 4))

# 2. Gradients
erreurs = A - y               # a - y pour chaque exemple
dw = X.T @ erreurs / m        # (1/m) X^T (A - y)
db = np.mean(erreurs)         # (1/m) somme des erreurs
print("Question 2 : gradients")
print("  erreurs A - y =", erreurs, "  dw =", dw, "  db =", db)

# 3. Mise à jour (un pas dans le sens de la descente)
w = w - eta * dw
b = b - eta * db
print("Question 3 : mise à jour")
print("  nouveau w =", w, "  nouveau b =", b)

# 4. Nouveau forward pass et nouveau coût
Z_new = X @ w + b
A_new = sigmoid(Z_new)
print("Question 4 : nouveau forward")
print("  z =", Z_new, "  A =", A_new, "  J =", round(cout(A_new, y), 4))

## 💻 Bonus : et si on continue ?

L'entraînement d'un modèle, ce n'est rien d'autre que **répéter ce pas** : forward, gradients, mise à jour, encore et encore. Voici 20 pas sur le même petit dataset :

In [ ]:
w, b = np.array([0.0, 0.0]), 0.0       # on repart du début
historique = []

for etape in range(20):
    A = sigmoid(X @ w + b)                        # 1. forward
    historique.append(cout(A, y))                 #    on garde le coût
    dw, db = X.T @ (A - y) / m, np.mean(A - y)    # 2. gradients
    w, b = w - eta * dw, b - eta * db             # 3. un pas vers le bas

print("Coût aux étapes 0, 1, 2, 5, 19 :", [float(round(historique[k], 3)) for k in [0, 1, 2, 5, 19]])
print("Poids finaux : w =", w, " b =", round(b, 3))
print("Prédictions finales :", sigmoid(X @ w + b), " (vraies étiquettes :", y, ")")

plt.figure(figsize=(6, 3.5))
plt.plot(historique, marker="o")
plt.xlabel("étape de descente de gradient")
plt.ylabel("coût J")
plt.title("Le coût baisse à chaque pas")
plt.grid(alpha=0.3)
plt.show()

> ✅ **À retenir (exercice 3)**
>
> - Gradient d'un exemple : $(a - y)\,x$ pour les poids, $(a - y)$ pour le biais. **Erreur × entrée.**
> - Sur le dataset : $dw = \frac{1}{m}X^\top(A - y)$ et $db = \frac{1}{m}\sum(a - y)$.
> - Mise à jour : $w \leftarrow w - \eta\,dw$, $b \leftarrow b - \eta\,db$. On va à l'opposé de la pente.
> - Un seul pas a fait passer $J$ de 0.693 à 0.257 : chaque prédiction s'est rapprochée de sa vraie étiquette.

---
# Bonus : PyTorch retrouve les mêmes chiffres (slide 21)

Le cours reprend l'exemple de l'exercice 1 ($w = (2, -1)$, $b = -1$, $x = (1, 3)$) en ajoutant la vraie étiquette $y = 1$. D'après la formule de l'exercice 3, le gradient vaut :

$$\frac{\partial \mathcal{L}}{\partial w} = (a - y)\,x = (0.119 - 1) \times (1, 3) = (-0.881,\ -2.642) \qquad \frac{\partial \mathcal{L}}{\partial b} = a - y = -0.881$$

PyTorch sait calculer ce gradient **tout seul** (c'est l'**autograd**) : on lui décrit le calcul, `loss.backward()` applique la règle de la chaîne à l'envers et range le résultat dans `w.grad`. Si PyTorch n'est pas installé, la cellule affiche seulement le calcul NumPy (pour l'installer : `pip install torch`, il sera nécessaire pour la partie C du TP02).

In [ ]:
w1, b1 = np.array([2.0, -1.0]), -1.0
x1, y1 = np.array([1.0, 3.0]), 1.0

a1 = sigmoid(w1 @ x1 + b1)
print("A la main (NumPy) :  dL/dw =", (a1 - y1) * x1, "  dL/db =", round(a1 - y1, 4))

try:
    import torch
    w_t = torch.tensor([2.0, -1.0], requires_grad=True)   # PyTorch enregistre les calculs faits avec w
    b_t = torch.tensor(-1.0, requires_grad=True)
    x_t, y_t = torch.tensor([1.0, 3.0]), torch.tensor(1.0)
    a_t = torch.sigmoid(w_t @ x_t + b_t)                   # forward pass
    loss = -(y_t * torch.log(a_t) + (1 - y_t) * torch.log(1 - a_t))
    loss.backward()                                         # règle de la chaîne automatique
    print("Autograd (PyTorch) : dL/dw =", w_t.grad.numpy(), "  dL/db =", round(b_t.grad.item(), 4))
except ImportError:
    print("PyTorch n'est pas installé : seul le calcul NumPy est affiché.")

---
# 📝 Fiche résumé de la session 2

| Étape | Formule | En une phrase |
|---|---|---|
| Score | $z = w^\top x + b$ | somme pondérée des entrées, plus le biais |
| Probabilité | $a = \sigma(z) = \frac{1}{1 + e^{-z}}$ | on écrase $z$ entre 0 et 1 |
| Décision | classe 1 si $a \ge 0.5$, c'est-à-dire si $z \ge 0$ | le signe de $z$ suffit |
| Loss | $\mathcal{L} = -[y\log a + (1 - y)\log(1 - a)]$ | punit fort les erreurs confiantes |
| Coût | $J = \frac{1}{m}\sum_i \mathcal{L}^{(i)}$ | la moyenne des loss |
| Dérivée de la sigmoïde | $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$ | maximum 0.25, atteint en $z = 0$ |
| Gradients | $dw = \frac{1}{m}X^\top(A - y)$, $db = \frac{1}{m}\sum_i (a^{(i)} - y^{(i)})$ | erreur × entrée |
| Mise à jour | $w \leftarrow w - \eta\,dw$, $b \leftarrow b - \eta\,db$ | un pas dans le sens de la descente |

**La limite de ce modèle** : la frontière de décision $w^\top x + b = 0$ reste une **droite**. Pour des données qui ne se séparent pas avec une droite (les « moons », le XOR), il faudra combiner plusieurs neurones : c'est le sujet de la session 3.